# 02 · Train — embedding engine → gridded-Argo pretrain → GLORYS fine-tune
Runtime: **GPU** (T4 works; A100/L4 ≈ 3× faster). Checkpoints go to Drive every epoch and training resumes.

In [ ]:
# ── Setup: Drive (data) + GitHub (code) + deps ──────────────────────────────
from google.colab import drive, userdata
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/OceanEmbed'          # data, checkpoints, outputs live here
REPO = '/content/Sylithe_Ocean_Embedding'
import os, sys; os.makedirs(ROOT, exist_ok=True)
if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/karan27-dev/Sylithe_Ocean_Embedding {REPO}
sys.path.insert(0, REPO)
for m in [m for m in sys.modules if m.startswith('oceanembed')]:
    del sys.modules[m]                              # re-running this cell picks up freshly pulled code
!git -C {REPO} log --oneline -1
# Colab already ships xarray, dask, netCDF4 and earthengine-api. Install only what is missing, one
# package at a time, so pip never has to search a large version space (that is what hangs).
!pip install -q --upgrade-strategy only-if-needed copernicusmarine
!pip install -q --upgrade-strategy only-if-needed earthaccess
!pip install -q --upgrade-strategy only-if-needed zarr
import importlib
for m in ['xarray', 'zarr', 'dask', 'netCDF4', 'ee', 'copernicusmarine', 'earthaccess']:
    print(f'{m:17s}', getattr(importlib.import_module(m), '__version__', 'ok'))

In [ ]:
import torch, shutil, numpy as np
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: Runtime → Change runtime type')
from oceanembed import config as C, dataset as D, train as TR
# Local copy: Drive reads per sample are slow, local SSD is fast
for s in ['inputs.zarr', 'target.zarr', 'inputs_monthly.zarr', 'argo_monthly.zarr']:
    if os.path.exists(f'{ROOT}/{s}') and not os.path.exists(f'/content/{s}'):
        shutil.copytree(f'{ROOT}/{s}', f'/content/{s}')
STATS = f'{ROOT}/stats.npz'
if not os.path.exists(STATS): D.compute_stats('/content/inputs.zarr', '/content/target.zarr', STATS)
S = D.load_stats(STATS)
cfg = C.TrainConfig(window=15, base=32, batch_size=8)
CK = f'{ROOT}/checkpoints/{cfg.arch}_w{cfg.window}'

## Stage 0 · Self-supervised embedding engine (masked surface autoencoder, no labels)

In [ ]:
mk = lambda period, train: D.SurfaceWindows('/content/inputs.zarr', None, S, period, cfg.window, train=train,
                                            var_dropout=cfg.var_dropout if train else 0, patch_mask=0.5, need_target=False)
mae, net = TR.new_models(cfg)
TR.fit(mae, mk(C.TRAIN, True), mk(C.VAL, False), cfg, 'ssl', CK, cfg.epochs_ssl)
TR.load_encoder_from_ssl(net, f'{CK}/ssl_best.pt')

## Stage 1 · Gridded-Argo monthly pretrain (skipped automatically if notebook 01 had no files)

In [ ]:
if os.path.exists('/content/argo_monthly.zarr'):
    Sm = dict(S)   # same normalisation; monthly samples use the mid-month climatology
    mkm = lambda p, tr: D.SurfaceWindows('/content/inputs_monthly.zarr', '/content/argo_monthly.zarr', Sm, p,
                                         cfg.window, train=tr, var_dropout=cfg.var_dropout if tr else 0)
    TR.fit(net, mkm(C.TRAIN, True), mkm(C.VAL, False), cfg, 'argo', CK, cfg.epochs_argo, depth_std=S['depth_std'])
    TR.load_weights(net, f'{CK}/argo_best.pt')

## Stage 2 · GLORYS daily fine-tune — all layers trainable (the paper's best strategy)

In [ ]:
mkd = lambda p, tr: D.SurfaceWindows('/content/inputs.zarr', '/content/target.zarr', S, p, cfg.window,
                                     train=tr, var_dropout=cfg.var_dropout if tr else 0)
TR.fit(net, mkd(C.TRAIN, True), mkd(C.VAL, False), cfg, 'glorys', CK, cfg.epochs_glorys, depth_std=S['depth_std'])

In [ ]:
# Validation skill per depth (vs GLORYS 2022)
import pandas as pd
v = torch.load(f'{CK}/glorys_best.pt', weights_only=False)['val']['per_depth']
pd.DataFrame(v).T.round(3)

## Baseline · faithful Attention 3D U-Net++ (Wang et al., ESSD 2026)
Same data, no embedding pretraining. Needed for the "ours vs published method" table.

In [ ]:
cfg3 = C.TrainConfig(window=15, base=16, batch_size=2, arch='attn_unetpp3d')
_, net3 = TR.new_models(cfg3)
CK3 = f'{ROOT}/checkpoints/attn_unetpp3d_w15'
TR.fit(net3, mkd(C.TRAIN, True), mkd(C.VAL, False), cfg3, 'glorys', CK3, cfg3.epochs_glorys, depth_std=S['depth_std'])

## Ablation · input window length (paper: gain saturates ≈ 26 d)

In [ ]:
for w in [1, 7, 26]:
    c = C.TrainConfig(window=w, epochs_glorys=20)
    _, n = TR.new_models(c); TR.load_encoder_from_ssl(n, f'{CK}/ssl_best.pt')   # encoder is window-agnostic
    mk_w = lambda p, tr: D.SurfaceWindows('/content/inputs.zarr', '/content/target.zarr', S, p, w, train=tr,
                                          var_dropout=c.var_dropout if tr else 0)
    print(w, TR.fit(n, mk_w(C.TRAIN, True), mk_w(C.VAL, False), c, 'glorys', f'{ROOT}/checkpoints/ablate_w{w}',
                    c.epochs_glorys, depth_std=S['depth_std']))